# Discover projects

If your organisation has a Breathe Discover account, you can find, create and edit Discover projects from Python, and run a project's KPIs on a cell or a virtual battery. A project bundles a design space (the designs to explore) with a KPI set (what to compute for each design).

Projects are looked up by name or uuid. Names are unique within an organisation; if a name matches projects from more than one organisation you can see, your own organisation's is used, otherwise pass the uuid.

In [ ]:
from breathe_simulate import api_interface as api

## Find a project

In [ ]:
projects = api.list_projects()
[p["name"] for p in projects]

In [ ]:
project = api.get_project(projects[0]["uuid"])
project["name"], project["kpi_set_uuid"]

## Design spaces and KPI sets

These are the building blocks you can link to a project.

In [ ]:
[ds["name"] for ds in api.list_design_spaces()]

In [ ]:
[ks["name"] for ks in api.list_kpi_sets()]

## Run a project's KPIs

Each project has a curated set of KPIs, each with fixed settings (C-rate, temperatures, cut-offs). List them, then run one on a cell, with design changes if you like, or on a Discover virtual battery (`virtual_battery_uuid=...`). Results come back to you per design; they are not saved to the project. KPIs with `returns_file` save a file (an ECM `.mat`, a lookup-table zip) to `output_dir`.

In [ ]:
kpis = api.list_kpis(project["uuid"])
[(k["name"], k["settings"]) for k in kpis]

In [ ]:
# a quick KPI that returns values, not a file
kpi = next((k for k in kpis if k["name"] == "eqm_kpis"), None) or next(
    k for k in kpis if not k["returns_file"]
)
cell = api.get_batteries()[0]
thickness = api.get_design_parameters(cell)["cathodeThickness_um"]
out = api.run_kpi(
    project["uuid"],
    kpi["name"],
    base_battery=cell,
    designs=[{"designName": "thicker cathode", "cathodeThickness_um": 1.1 * thickness}],
)
out["thicker cathode"]

## Create and edit a project

This needs permission to create and edit Discover projects in your organisation. A new project always gets its own copy of a KPI set, so editing its KPIs later never changes anyone else's. Without `kpi_set` or `clone_from`, it copies Breathe's standard KPI set. A design space is linked, not copied. Pass `clone_from="<project>"` to start from one of your projects: it links its design space and copies its KPI set.

New projects are private to your organisation.

In [ ]:
new = api.create_project("My SDK project", description="Created from the Python SDK")
new["uuid"], new["kpi_set_uuid"]

Only the arguments you pass change. You can only edit projects your organisation owns.

In [ ]:
api.update_project("My SDK project", description="Second pass")